# 01 - RQ2 experiments (scripted agents, simulated server)
Runs the approved design (`docs/stage2_design_proposal.md`). Everything is **resumable**: if Colab disconnects, re-run the cells and it continues.
Order: setup -> dev (5,520 runs) -> select -> test (1,080 runs). CPU is enough (no GPU needed).

In [ ]:
REPO_URL = "https://github.com/notanut/eem-acc-skripsi.git"
REPO_DIR = "/content/eem-acc-skripsi"
DRIVE_ROOT = "/content/drive/MyDrive/Skripsi"
SEED = 42

import base64, os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
def git(*args, cwd=None):
    cmd = ["git"]
    if token:
        cmd += ["-c", "http.extraheader=Authorization: Basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()]
    subprocess.run(cmd + list(args), cwd=cwd, check=True)
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    git("pull", "--ff-only", cwd=REPO_DIR)
else:
    git("clone", REPO_URL, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt", "-e", REPO_DIR], check=True)
os.environ["EEM_ACC_DRIVE_ROOT"] = DRIVE_ROOT
from eem_acc.utils import Paths, set_seed
set_seed(SEED); paths = Paths().ensure(); print(paths)

In [ ]:
# Dev phase: 5,520 runs. Safe to interrupt and re-run (continues where it stopped).
!cd {REPO_DIR} && PYTHONPATH=src python experiments/run_rq2.py --phase dev --root {DRIVE_ROOT}

In [ ]:
# Select the frozen configuration from the dev results (refuses if the dev phase is incomplete).
!cd {REPO_DIR} && PYTHONPATH=src python experiments/run_rq2.py --phase select --root {DRIVE_ROOT}
# The frozen config is copied to Drive (results/rq2/rq2_frozen.json). Keep a copy in the repo (configs/rq2_frozen.json) BEFORE the test phase.

In [ ]:
# Test phase: 1,080 runs with the frozen configuration and fresh seeds (100-109).
!cd {REPO_DIR} && PYTHONPATH=src python experiments/run_rq2.py --phase test --root {DRIVE_ROOT}